# Local TikZ n-gram repetition

This notebook detects repetition **inside each individual TikZ/LaTeX sample**.

It does **not** compare samples against the complete dataset.

Pipeline:

```text
one TikZ sample
→ remove comments
→ normalize numbers to NUM
→ tokenize
→ build 1-, 2-, 3-, 4-grams
→ count repeated n-grams inside this sample
→ compute repetition coverage
→ low / medium / high / critical
```

The original code is written into `low/`, `medium/`, or `critical/`. A CSV with metrics and the most frequent local n-grams is written as well.


In [9]:
from collections import Counter
from pathlib import Path
import json
import re
import shutil

import pandas as pd
import pyarrow.parquet as pq
from tqdm.auto import tqdm


## Configuration

In [ ]:
PARQUET_DIR = Path("/home/jonas/Datasets/TikZ/DatikZ-v4-raw-test")
OUTPUT_DIR = Path("./local_repetition_test")

CODE_COLUMNS = (
    "reference_code",
    "code",
    "tikz_code",
)

NGRAM_ORDERS = (2, 3, 4, 6)

NGRAM_WEIGHTS = {
    2: 0.35,
    3: 0.30,
    4: 0.20,
    6: 0.20,
}

# An n-gram must occur this often inside the SAME sample.
MIN_REPEAT_COUNT = 2

MEDIUM_THRESHOLD = 0.25
HIGH_THRESHOLD = 0.45
CRITICAL_THRESHOLD = 0.65

TOP_LOCAL_NGRAMS = 20
BATCH_SIZE = 2048
OVERWRITE = True


## Normalize and tokenize

In [11]:
NUMBER_RE = re.compile(
    r"""
    (?<![A-Za-z])
    [-+]?
    (?:
        \d+\.\d*
        |
        \.\d+
        |
        \d+
    )
    (?:[eE][-+]?\d+)?
    """,
    re.VERBOSE,
)

TOKEN_RE = re.compile(
    r"""
    \\[A-Za-z@]+
    | NUM
    | [A-Za-z_][A-Za-z0-9_]*
    | --|->|<-|<->
    | [^\s]
    """,
    re.VERBOSE,
)


def strip_comments(code: str) -> str:
    return "\n".join(
        re.sub(r"(?<!\\)%.*$", "", line)
        for line in str(code or "").splitlines()
    )


def normalize(code: str) -> str:
    return NUMBER_RE.sub("NUM", strip_comments(code))


def tokenize(code: str) -> list[str]:
    return TOKEN_RE.findall(normalize(code))


## Local n-gram repetition

In [12]:
def make_ngrams(tokens: list[str], n: int) -> list[tuple[str, ...]]:
    return [
        tuple(tokens[i:i + n])
        for i in range(len(tokens) - n + 1)
    ]


def local_ngram_metrics(tokens: list[str], n: int) -> dict:
    grams = make_ngrams(tokens, n)

    if not grams:
        return {
            "coverage": 0.0,
            "max_frequency": 0,
            "counts": Counter(),
        }

    counts = Counter(grams)
    repeated_occurrences = sum(
        count
        for count in counts.values()
        if count >= MIN_REPEAT_COUNT
    )

    return {
        "coverage": repeated_occurrences / len(grams),
        "max_frequency": max(counts.values()),
        "counts": counts,
    }


def repetition_metrics(code: str) -> dict:
    tokens = tokenize(code)

    if not tokens:
        return {
            "repetition_score": 0.0,
            "repetition_class": "low",
            "token_count": 0,
            **{f"ngram_{n}_coverage": 0.0 for n in NGRAM_ORDERS},
            "top_local_ngrams": "[]",
        }

    score = 0.0
    output = {}
    repeated = []

    for n in NGRAM_ORDERS:
        metrics = local_ngram_metrics(tokens, n)
        coverage = metrics["coverage"]
        score += NGRAM_WEIGHTS[n] * coverage
        output[f"ngram_{n}_coverage"] = coverage

        for gram, count in metrics["counts"].most_common():
            if count < MIN_REPEAT_COUNT:
                break
            repeated.append({
                "n": n,
                "count": count,
                "ngram": " ".join(gram),
            })

    repeated.sort(key=lambda x: (x["count"], x["n"]), reverse=True)

    if score < MEDIUM_THRESHOLD:
        repetition_class = "low"
    elif score < HIGH_THRESHOLD:
        repetition_class = "medium"
    elif score < CRITICAL_THRESHOLD:
        repetition_class = "high"
    else:
        repetition_class = "critical"

    return {
        "repetition_score": score,
        "repetition_class": repetition_class,
        "token_count": len(tokens),
        **output,
        "top_local_ngrams": json.dumps(
            repeated[:TOP_LOCAL_NGRAMS],
            ensure_ascii=False,
        ),
    }


## Quick test

In [13]:
example = r"""
\begin{tikzpicture}
    \draw (0,0) -- (1,1);
    \draw (1,1) -- (2,2);
    \draw (2,2) -- (3,3);
    \draw (3,3) -- (4,4);
    \draw (4,4) -- (5,5);
    \draw (5,5) -- (6,6);
\end{tikzpicture}
"""

example2 = r"""
\documentclass[tikz]{standalone}

\usepackage[utf8]{inputenc}
\usepackage{amsmath} %Faire des mathématiques
\usepackage{amssymb,amsthm} %Faire des mathématiques
\usepackage{color}%Pour colorer des figures
\usepackage{tikz} % Pour dessiner des graphiques
\usetikzlibrary{matrix,arrows} %Pour rajouter des dessins de flèches dans tikz
\usepackage{braids} %Pour rajouter des tresses dans tikz

\begin{document}

\begin{tikzpicture}
\matrix[ampersand replacement=\&,matrix of math nodes,row sep=1cm,column sep=1cm]{
|(C)| F(X) \& |(D)| G(X) \\
|(C')| F(Y) \& |(D')| G(Y) \\
};
\begin{scope}[every node/.style={pos=0.5,font=\scriptsize}]
\draw[->] (C) -- (D) node[auto=left] {$\rho_{X}$};
\draw[->] (C') -- (D') node[auto=right] {$\rho_{Y}$} ;
\draw[->] (C) -- (C') node[auto=right] {$F(f)$};
\draw[->] (D) -- (D') node[auto=left] {$\nu_{f}G(f)$};
\end{scope}
\end{tikzpicture}

\end{document}"""

example3=r"""
\documentclass[tikz]{standalone}

\usepackage{amsmath}
\usepackage{amssymb}
\usepackage{color}
\usepackage[dvipsnames]{xcolor}
\usepackage[compat=1.1.0]{tikz-feynman}
\usepackage{tikz-feynman,contour}

\begin{document}

\begin{tikzpicture}
		\begin{feynman}
		\vertex(a) at (-1.5,0);
		\vertex(b) at (0,0);
		\vertex(c) at (-0.8,0.5);
		\vertex(d) at (-0.8,-0.5);
		\diagram* {(a) -- (b), (b) -- (c),(b) -- (d)};
		\end{feynman}
		\end{tikzpicture}

\end{document}"""

display(pd.DataFrame([repetition_metrics(example)]))


,repetition_score,repetition_class,token_count,ngram_2_coverage,ngram_3_coverage,ngram_4_coverage,ngram_6_coverage,top_local_ngrams
0,0.973071,critical,86,0.952941,0.928571,0.903614,0.901235,"[{""n"": 4, ""count"": 12, ""ngram"": ""( NUM , NUM""}..."


In [20]:
def show_local_ngrams(code: str, n: int, top_k: int = 30):
    counts = Counter(make_ngrams(tokenize(code), n))
    display(pd.DataFrame([
        {"count": count, "ngram": " ".join(gram)}
        for gram, count in counts.most_common(top_k)
    ]))

show_local_ngrams(example, n=5)


,count,ngram
0,12,"( NUM , NUM )"
1,6,"\draw ( NUM , NUM"
2,6,"NUM , NUM ) --"
3,6,", NUM ) -- ("
4,6,NUM ) -- ( NUM
5,6,") -- ( NUM ,"
6,6,"-- ( NUM , NUM"
7,6,"NUM , NUM ) ;"
8,5,", NUM ) ; \draw"
9,5,NUM ) ; \draw (


## Process Parquet files

In [15]:
def find_code_column(parquet_file: Path) -> str:
    columns = set(pq.ParquetFile(parquet_file).schema.names)

    for column in CODE_COLUMNS:
        if column in columns:
            return column

    raise ValueError(
        f"No code column found in {parquet_file}. Tried: {CODE_COLUMNS}"
    )


def prepare_output():
    if OUTPUT_DIR.exists():
        if not OVERWRITE:
            raise FileExistsError(
                f"{OUTPUT_DIR} already exists. Set OVERWRITE=True to replace it."
            )
        shutil.rmtree(OUTPUT_DIR)

    for class_name in ("low", "medium", "high", "critical"):
        (OUTPUT_DIR / class_name).mkdir(parents=True, exist_ok=True)


parquet_files = sorted(PARQUET_DIR.rglob("*.parquet"))

if not parquet_files:
    raise FileNotFoundError(f"No Parquet files found in {PARQUET_DIR}")

prepare_output()

rows = []
sample_index = 0

for parquet_file in tqdm(parquet_files, desc="Parquet files"):
    code_column = find_code_column(parquet_file)
    parquet = pq.ParquetFile(parquet_file)

    for batch in parquet.iter_batches(
        columns=[code_column],
        batch_size=BATCH_SIZE,
    ):
        for code in batch.column(0).to_pylist():
            code = str(code or "")
            metrics = repetition_metrics(code)
            class_name = metrics["repetition_class"]

            output_file = (
                OUTPUT_DIR
                / class_name
                / f"{sample_index:09d}.txt"
            )
            output_file.write_text(code, encoding="utf-8")

            rows.append({
                "sample_index": sample_index,
                "source_file": str(parquet_file),
                "output_file": str(output_file),
                **metrics,
            })
            sample_index += 1

results = pd.DataFrame(rows)
results.to_csv(
    OUTPUT_DIR / "local_repetition_results.csv",
    index=False,
)

print(f"Processed: {len(results):,}")
display(
    results["repetition_class"]
    .value_counts()
    .reindex(["low", "medium", "high", "critical"], fill_value=0)
    .to_frame("samples")
)


Parquet files:   0%|          | 0/1 [00:00<?, ?it/s]

Parquet files: 100%|██████████| 1/1 [00:06<00:00,  6.51s/it]

Processed: 5,000


,samples
repetition_class,
low,218
medium,534
high,1134
critical,3114


## Inspect result distribution

In [16]:
display(
    results.groupby("repetition_class")[[
        "repetition_score",
        "ngram_2_coverage",
        "ngram_3_coverage",
        "ngram_4_coverage",
        "ngram_5_coverage",
    ]].describe()
)

display(
    results.sort_values(
        "repetition_score",
        ascending=False,
    ).head(50)
)


KeyError: "Columns not found: 'ngram_5_coverage'"